In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  x=missing_values[missing_values > 0]
  print(len(x))
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)



In [ ]:
for col in df.columns:
    df[col] = df[col].fillna(df[col].mean())


In [ ]:
df.isnull().sum().sum()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print(categorical_cols)


#No categorical columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns
import matplotlib.pyplot as plt
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, 'Target')


## There is an imbalance in the data


In [ ]:
# Task 1: Write your code here:
X= df.drop('Target',axis=1)
y=df['Target']

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,  f1_score
n_splits = 5 # K
from catboost import CatBoostClassifier

model = CatBoostClassifier( verbose=0,
      n_estimators=320,
      max_depth=4)

results = {}
results['CatBoostClassifier'] = {'accuracy': [], 'f1': []}

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)

  f1 = f1_score(y_test, y_pred, zero_division=0)

  results["CatBoostClassifier"]['accuracy'].append(accuracy)
  results["CatBoostClassifier"]['f1'].append(f1)


In [ ]:
print(sum(results["CatBoostClassifier"]['accuracy'])/len(results["CatBoostClassifier"]['accuracy']))
print(sum(results["CatBoostClassifier"]['f1'])/len(results["CatBoostClassifier"]['f1']))

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
feature_importance['feature'][0]

In [ ]:
# Task Bonus: Write your code here:
X= df[['P_2']]

In [ ]:
results_single_f = {}
results_single_f['CatBoostClassifier'] = {'accuracy': [], 'f1': []}

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate


    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)

  f1 = f1_score(y_test, y_pred, zero_division=0)

  results_single_f["CatBoostClassifier"]['accuracy'].append(accuracy)
  results_single_f["CatBoostClassifier"]['f1'].append(f1)

In [ ]:

print(f"\n{'CatBoostClassifier'}:")
print(f"  Accuracy:  {np.mean(results['CatBoostClassifier']['accuracy']):.4f}")
print(f"  Precision: {np.mean(results["CatBoostClassifier"]['f1']):.4f}")


print(f"\n{'CatBoostClassifier with golden feature'}:")
print(f"  Accuracy:  {np.mean(results_single_f['CatBoostClassifier']['accuracy']):.4f}")
print(f"  Precision: {np.mean(results_single_f["CatBoostClassifier"]['f1']):.4f}")


In [ ]:
# This feature alone reached an accuracy of 0.79 and f1 of 0.5846